<a href="https://colab.research.google.com/github/devjangid2005bmr/next_word_predictor/blob/main/Sentence_Completion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
df = pd.read_csv("/content/qoute_dataset.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [7]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [8]:
df.shape

(3038, 2)

In [9]:
quotes = df['quote']

In [10]:
quotes

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."
...,...
3033,The past beats inside me like a second heart.
3034,"Damn, Claire. Warn a guy before you do a face-..."
3035,"Can you be a girl for a few seconds?""""I'm alwa..."
3036,That's what fiction is for. It's for getting a...


PREPROCESS DATA

In [11]:
quotes = quotes.str.lower()

In [12]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))


In [13]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


TOKENIZATION

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [15]:
vocab_size=1000

tokanizer = Tokenizer(num_words=vocab_size)
tokanizer.fit_on_texts(quotes)

In [16]:
word_index = tokanizer.word_index
print(len(word_index))

list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [17]:
sequence = tokanizer.texts_to_sequences(quotes)

In [18]:
for i in range(3):
  print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [19]:
for i in range(3):
  print(sequence[i])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70]
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70]
[14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1, 101, 7, 29, 329, 126, 7, 5]


In [20]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [21]:

print(len(X))
print(len(y))

71115
71115


### Padding Sequences

To prepare the sequences for a neural network, it's common to pad them so that they all have the same length. This is done by adding zeros to the beginning or end of the sequences.

In [22]:
max_len = max(len(x) for x in X)
print(max_len)

578


In [23]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [24]:
X_padded = pad_sequences(X , maxlen=max_len , padding= 'pre')

In [25]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [26]:
y = np.array(y)

In [27]:
X_padded.shape

(71115, 578)

### One-Hot Encoding the Target Variable

For classification tasks, it's necessary to convert the integer-encoded target variable (`y`) into a one-hot encoded format. This creates a binary column for each possible class, with a '1' indicating the presence of that class.

In [28]:
from tensorflow.keras.utils import to_categorical

y_one_hot = to_categorical(y, num_classes=vocab_size)


In [29]:
y.shape

(71115,)

In [30]:
y_one_hot.shape

(71115, 1000)

In [31]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense , SimpleRNN

In [32]:
embedding_dim = 50
rnn_units = 128


In [33]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)

)

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))
#

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [35]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [36]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
    )

lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [37]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [38]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [39]:
epochs = 10
batch_size = 128


In [40]:
history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 31s 58ms/step - accuracy: 0.0523 - loss: 5.6038 - val_accuracy: 0.0821 - val_loss: 5.3770
Epoch 2/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 23s 51ms/step - accuracy: 0.0935 - loss: 5.4231 - val_accuracy: 0.0975 - val_loss: 5.3716
Epoch 3/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.1003 - loss: 5.3168 - val_accuracy: 0.1101 - val_loss: 5.1741
Epoch 4/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.1142 - loss: 5.0604 - val_accuracy: 0.1102 - val_loss: 5.1027
Epoch 5/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.1175 - loss: 5.0181 - val_accuracy: 0.1117 - val_loss: 5.0758
Epoch 6/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.1233 - loss: 4.9156 - val_accuracy: 0.1180 - val_loss: 5.0208
Epoch 7/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 23s 52ms/step - accuracy: 0.1050 - loss: 5.2737 - val_accuracy: 0.1091 - val_loss: 5.1300
Epoch 8/10
445/445 ━━━━━━━━━━━━━━━━━━━━ 22s 50ms/step - accuracy: 0.1089 - loss: 5.1780 - 

In [41]:
rnn_model.save("rnn_model.h5")

In [42]:
index_to_word = {}

for word , index in word_index.items():
  index_to_word[index] = word

In [52]:
from re import VERBOSE
import numpy as np

def predictor(model, tokanizer, text, next_words=1):
  text = text.lower()

  # Convert text to sequence
  seq = tokanizer.texts_to_sequences([text])[0]
  # FIX: Use the local 'seq' variable, not the global 'sequence'
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  # Predict
  pred = model.predict(seq, verbose=0)
  pred_word_index = np.argmax(pred)

  return index_to_word.get(pred_word_index, "<OOV>")

In [60]:
seed_text = "what are you"
next_word = predictor(rnn_model , tokanizer , seed_text , max_len)
print(seed_text , next_word)

what are you are


In [67]:
def generate_text(model, tokanizer, seed_text, max_len , n_words):
  for _ in range(n_words):
    next_word = predictor(model, tokanizer, seed_text, max_len)
    if next_word is None:
      break
    seed_text += " " + next_word
  return seed_text

In [69]:
seed = "are you a"
# Renamed the variable to avoid shadowing the function name
generated_result = generate_text(rnn_model, tokanizer, seed, max_len, 10)
print(generated_result)

are you a heart and the world is not the of the world


In [70]:
import pickle

with open("tokanizer.pkl" , "wb") as f:
  pickle.dump(tokanizer , f)



In [71]:
with open("max_len.pkl" , "wb") as f:
  pickle.dump(max_len , f)